In [8]:
import json

products_data = [
  {
    "id": 1,
    "name": "HP Pavilion 14",
    "brand": "HP",
    "category": "laptop",
    "price": 72000,
    "rating": 4.3,
    "specs": {
      "ram": "16GB",
      "processor": "Intel i7 12th Gen",
      "storage": "512GB SSD",
      "display": "14 inch FHD",
      "gpu": "Intel Iris Xe"
    },
    "tags": ["video editing", "student", "portable"]
  },
  {
    "id": 2,
    "name": "Dell Inspiron 15",
    "brand": "Dell",
    "category": "laptop",
    "price": 68000,
    "rating": 4.2,
    "specs": {
      "ram": "16GB",
      "processor": "Intel i5 12th Gen",
      "storage": "512GB SSD",
      "display": "15.6 inch FHD",
      "gpu": "NVIDIA MX350"
    },
    "tags": ["office", "light editing"]
  },
  {
    "id": 3,
    "name": "Apple MacBook Air M1",
    "brand": "Apple",
    "category": "laptop",
    "price": 85000,
    "rating": 4.8,
    "specs": {
      "ram": "8GB",
      "processor": "Apple M1",
      "storage": "256GB SSD",
      "display": "13.3 inch Retina",
      "gpu": "Integrated"
    },
    "tags": ["video editing", "premium"]
  },
  {
    "id": 4,
    "name": "iPhone 13",
    "brand": "Apple",
    "category": "mobile",
    "price": 65000,
    "rating": 4.7,
    "specs": {
      "ram": "4GB",
      "processor": "A15 Bionic",
      "storage": "128GB",
      "display": "6.1 inch OLED",
      "camera": "12MP Dual"
    },
    "tags": ["camera", "premium"]
  },
  {
    "id": 5,
    "name": "Samsung Galaxy S21 FE",
    "brand": "Samsung",
    "category": "mobile",
    "price": 50000,
    "rating": 4.4,
    "specs": {
      "ram": "8GB",
      "processor": "Exynos 2100",
      "storage": "128GB",
      "display": "6.4 inch AMOLED",
      "camera": "12MP Triple"
    },
    "tags": ["camera", "gaming"]
  }
]

with open("electronics_catalog.json", "w") as f:
    json.dump(products_data, f, indent=2)


In [2]:
pip install sentence-transformers faiss-cpu pandas streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 98.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 86.3 MB/s eta 0:00:00


In [3]:
import json
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

# Load model
model = SentenceTransformer('all-MiniLM-L6-v2')

class ElectronicsEngine:
    def __init__(self, data_path):
        with open(data_path, "r") as f:
            self.products = json.load(f)

        self.texts = [self._product_to_text(p) for p in self.products]
        self.embeddings = model.encode(self.texts)

        self.index = faiss.IndexFlatL2(self.embeddings.shape[1])
        self.index.add(np.array(self.embeddings))

    def _product_to_text(self, product):
        specs = " ".join(product["specs"].values())
        tags = " ".join(product["tags"])
        return f"{product['name']} {product['brand']} {specs} {tags}"

    def search(self, query, top_k=5, max_price=None):
        query_embedding = model.encode([query])
        distances, indices = self.index.search(query_embedding, top_k)

        results = []
        for idx in indices[0]:
            product = self.products[idx]

            if max_price and product["price"] > max_price:
                continue

            score = self._score(product, query)
            results.append((product, score))

        results = sorted(results, key=lambda x: x[1], reverse=True)
        return [r[0] for r in results]

    def _score(self, product, query):
        score = 0

        # rating weight
        score += product["rating"] * 2

        # price advantage (cheaper = better)
        score += 100000 / product["price"]

        # tag match
        for tag in product["tags"]:
            if tag.lower() in query.lower():
                score += 5

        return score

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [4]:
def compare_products(products):
    comparison = []

    for p in products:
        comparison.append({
            "Name": p["name"],
            "Price": p["price"],
            "Rating": p["rating"],
            "RAM": p["specs"].get("ram"),
            "Processor": p["specs"].get("processor"),
            "Storage": p["specs"].get("storage")
        })

    return comparison


def summarize_comparison(products):
    best = max(products, key=lambda x: x["rating"])

    summary = f"""
Best Overall: {best['name']}

Why:
- Highest rating: {best['rating']}
- Suitable for: {", ".join(best['tags'])}

Trade-offs:
"""
    for p in products:
        summary += f"- {p['name']} cheaper at ₹{p['price']}\n"

    return summary

In [5]:
def image_to_query(description):
    """
    Replace with Gemini / Llama Vision later
    """
    return description  # placeholder

In [9]:
engine = ElectronicsEngine("electronics_catalog.json")


def agent(query):
    query = query.lower()

    # Extract price constraint
    max_price = None
    if "under" in query:
        try:
            max_price = int(query.split("under")[1].strip().split()[0])
        except:
            pass

    # Compare flow
    if "compare" in query:
        results = engine.search(query, top_k=3)
        table = compare_products(results)
        summary = summarize_comparison(results)
        return table, summary

    # Default search
    results = engine.search(query, top_k=5, max_price=max_price)

    return results, "Top recommendations based on your query"

In [11]:
import streamlit as st

st.set_page_config(page_title="Electronics AI Assistant")

st.title("🛒 Electronics Shopping Agent")

query = st.text_input("Ask something (e.g. best laptop under 80000 for video editing)")

if query:
    results, message = agent(query)

    st.subheader("Results")

    if isinstance(results, list) and isinstance(results[0], dict):
        # product list
        for p in results:
            st.write(f"### {p['name']}")
            st.write(f"💰 ₹{p['price']} | ⭐ {p['rating']}")
            st.write(p["specs"])
            st.write("---")

    else:
        # comparison table
        st.table(results)
        st.write(message)

2026-03-29 08:53:10.087 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 08:53:10.088 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 08:53:10.245 
  command:

    streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py [ARGUMENTS]
2026-03-29 08:53:10.246 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 08:53:10.248 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 08:53:10.250 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-29 08:53:10.251 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when runn

In [13]:
!streamlit run /usr/local/lib/python3.12/dist-packages/colab_kernel_launcher.py




  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.83.214.14:8501

  Stopping...
  Stopping...
